In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
np.random.seed(42)

In [7]:
# upload  price data that had been saved in stocks_data_adj_close.csv into the file features_extraction.ipynb
stocks_data_adj_close= pd.read_csv("stocks_data_adj_close.csv")
stocks_data_adj_close = stocks_data_adj_close.set_index("Date")
tickers = stocks_data_adj_close.columns.tolist()

# calculate daily log-returns
log_returns = np.log(stocks_data_adj_close / stocks_data_adj_close.shift(1))
log_returns = log_returns.dropna()
stocks_data = stocks_data_adj_close[1:]

# set the index in datetime format
log_returns.index = pd.to_datetime(log_returns.index)

# create a column indicating the week it belongs to
log_returns['Week'] = log_returns.index.to_period('W')

# Group daily log-returns by week.
weekly_log_returns = log_returns.groupby('Week')

# create a list of DataFrames, which contain the log-returns for each week without the 'Week' column
weekly_log_returns = [group.drop(columns='Week') for _, group in weekly_log_returns]

# get an overview of weekly_log_returns
weekly_log_returns[0:2]

[             EXV1.DE   EXV2.DE   EXV3.DE   EXV5.DE   EXV6.DE
 Date                                                        
 2014-06-09  0.007305  0.008258  0.002483 -0.001895  0.007897
 2014-06-10 -0.005509 -0.002965  0.003889 -0.004562 -0.005738
 2014-06-11 -0.009727 -0.007949 -0.005661 -0.004774  0.000479
 2014-06-12 -0.001813  0.002989  0.003896  0.000000 -0.015942
 2014-06-13 -0.001913 -0.001992 -0.003541 -0.007301  0.002917,
              EXV1.DE   EXV2.DE   EXV3.DE   EXV5.DE   EXV6.DE
 Date                                                        
 2014-06-16 -0.008893 -0.009681 -0.000710 -0.003477  0.001456
 2014-06-17  0.003615 -0.000336  0.004958  0.006172  0.003871
 2014-06-18 -0.004340  0.008354  0.003878  0.000193  0.005058
 2014-06-19 -0.000338  0.002659  0.002460  0.007280  0.007897
 2014-06-20 -0.011571 -0.009001 -0.003516 -0.002676  0.002381]

In [8]:
# Function to calculate the optimal portfolio weights that maximize the Sharpe Ratio each week
def get_df_optimal_sharpe_ratio_weights(tickers, log_returns, risk_free_rate=0.02):
    cov_matrix = log_returns.cov() * 252

    # function to calculate the standard deviation
    def standard_deviation(weights, cov_matrix):
        variance = weights.T @ cov_matrix @ weights
        return np.sqrt(variance)

    # function to calculate expected return
    def expected_return(weights, log_returns):
        return np.sum(log_returns.mean() * weights) * 252

    # function to calculate Sharpe ratio
    def sharpe_ratio(weights, log_returns, cov_matrix, risk_free_rate):
        return (expected_return(weights, log_returns) - risk_free_rate) / standard_deviation(weights, cov_matrix)

    # function to calculate the opposite of the Sharpe ratio
    def neg_sharpe_ratio(weights, log_returns, cov_matrix, risk_free_rate):
        return -sharpe_ratio(weights, log_returns, cov_matrix, risk_free_rate)

    # set the constraints and limits for the portfolio weights (the sum equal to 1 and each weight plus 0.4)
    constraints = {'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}
    bounds = [(0, 0.4) for _ in range(len(tickers))]
    initial_weights = np.array([1/len(tickers)] * len(tickers))

    # perform the optimization: minimize the opposite of the Sharpe ratio and then maximize the Sharpe ratio
    optimized_results = minimize(neg_sharpe_ratio, initial_weights, args=(log_returns, cov_matrix, risk_free_rate),
                                 method='SLSQP', constraints=constraints, bounds=bounds)

    optimal_weights = optimized_results.x
    weights = [round(weight, 2) for weight in optimal_weights]

    return weights

In [9]:
# df with the optimal weights for each week that maximize the Sharpe Ratio
optimal_weights = []
for log_returns in weekly_log_returns:
    optimal_weights.append(get_df_optimal_sharpe_ratio_weights(tickers, log_returns, risk_free_rate=0.02))
df_optimal_weights = pd.DataFrame(optimal_weights, columns=tickers)
df_optimal_weights.to_csv("df_optimal_weights.csv", index=False)
df_optimal_weights.head(5)

,EXV1.DE,EXV2.DE,EXV3.DE,EXV5.DE,EXV6.DE
0,0.2,0.4,0.4,0.0,0.0
1,0.0,0.0,0.4,0.2,0.4
2,0.0,0.4,0.2,0.0,0.4
3,0.0,0.0,0.4,0.2,0.4
4,0.4,0.2,0.0,0.4,0.0


In [10]:
# df with the weights of an equally weighted portfolio (weights equally distributed over all securities that are part of the portfolio)
equally_weighted_portfolio_weights = []
for n in range(len(weekly_log_returns)):
    equally_weighted_portfolio_weights.append([1/len(tickers)] * len(tickers))
df_equally_weighted_portfolio_weights = pd.DataFrame(equally_weighted_portfolio_weights, columns=tickers)
df_equally_weighted_portfolio_weights.to_csv("df_equally_weighted_portfolio_weights.csv", index=False)
df_equally_weighted_portfolio_weights.head(5)

,EXV1.DE,EXV2.DE,EXV3.DE,EXV5.DE,EXV6.DE
0,0.2,0.2,0.2,0.2,0.2
1,0.2,0.2,0.2,0.2,0.2
2,0.2,0.2,0.2,0.2,0.2
3,0.2,0.2,0.2,0.2,0.2
4,0.2,0.2,0.2,0.2,0.2
